In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import accuracy_score
from PIL import Image
import pandas as pd
from sklearn.metrics import f1_score



In [2]:
# Device configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training on device:", DEVICE)

# Constants
BATCH_SIZE = 32
NUM_EPOCHS = 50
IMAGE_SIZE = 128
TRAIN_DIR = "potato_train/train"
TEST_DIR = "potato_test/potato_test"
CLASS_NAMES = sorted(os.listdir(TRAIN_DIR))
NUM_CLASSES = len(CLASS_NAMES)


# Transforms
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.2), ratio=(0.3, 3.3))

])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


# Dataset
full_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

val_dataset.dataset.transform = val_transform

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)


Training on device: cuda


In [3]:
# CNN Model
class PotatoCNN(nn.Module):
    def __init__(self):
        super(PotatoCNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * (IMAGE_SIZE // 16) * (IMAGE_SIZE // 16), 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, NUM_CLASSES)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        return self.fc_layers(x)


In [4]:
# Training Function
def train():
    model = PotatoCNN().to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

    best_f1 = 0.0
    patience = 5
    epochs_without_improvement = 0
    
    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0
        val_loss = 0.0
        correct_train = 0
        total_train = 0
        
        
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct_train += (predicted == labels).sum().item()
            total_train += labels.size(0)
            
        scheduler.step()
        
        train_loss = running_loss / len(train_loader)
        train_acc = correct_train / total_train

        # Validation
        model.eval()
        all_preds, all_labels = [], []
        
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, preds = torch.max(outputs, 1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                val_f1 = f1_score(all_labels, all_preds, average='macro')


        val_loss /= len(val_loader)
        val_acc = accuracy_score(all_labels, all_preds)
        print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | "
              f"Val F1 Score: {val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            epochs_without_improvement = 0
            torch.save(model.state_dict(), "LeModelB.pth")
            print("Best model saved with F1 score:", best_f1)
        else:
            epochs_without_improvement += 1
            print(f"No improvement for {epochs_without_improvement} epoch(s).")
            
        if epochs_without_improvement >= patience:
            print(f"Early stopping triggered after {patience} epochs without improvement.")
            break

    print("Training complete. Best F1 score:", best_f1)

train()


Epoch [1/50] Train Loss: 3.3817 | Train Acc: 0.2707 | Val Loss: 1.5730 | Val Acc: 0.4170 | Val F1 Score: 0.2973
Best model saved with F1 score: 0.2973121056379382
Epoch [2/50] Train Loss: 1.6055 | Train Acc: 0.3677 | Val Loss: 1.5074 | Val Acc: 0.4428 | Val F1 Score: 0.3266
Best model saved with F1 score: 0.32656948719928725
Epoch [3/50] Train Loss: 1.5726 | Train Acc: 0.3820 | Val Loss: 1.4520 | Val Acc: 0.4742 | Val F1 Score: 0.4108
Best model saved with F1 score: 0.410809955331129
Epoch [4/50] Train Loss: 1.5204 | Train Acc: 0.4139 | Val Loss: 1.4584 | Val Acc: 0.4373 | Val F1 Score: 0.3721
No improvement for 1 epoch(s).
Epoch [5/50] Train Loss: 1.4759 | Train Acc: 0.4388 | Val Loss: 1.4296 | Val Acc: 0.5000 | Val F1 Score: 0.4230
Best model saved with F1 score: 0.42299928748569343
Epoch [6/50] Train Loss: 1.4595 | Train Acc: 0.4360 | Val Loss: 1.4362 | Val Acc: 0.4926 | Val F1 Score: 0.3616
No improvement for 1 epoch(s).
Epoch [7/50] Train Loss: 1.4143 | Train Acc: 0.4845 | Val Los

In [5]:
"""Potato_Test to csv"""

def predict_and_generate_csv(model_path="LeModelB.pth"):
    model = PotatoCNN().to(DEVICE)
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.eval()

    transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    results = []
    image_filenames = (os.listdir(TEST_DIR))

    with torch.no_grad():
        for image_name in image_filenames:
            image_path = os.path.join(TEST_DIR, image_name)
            image = Image.open(image_path).convert("RGB")
            image = transform(image).unsqueeze(0).to(DEVICE)

            output = model(image)
            _, predicted = torch.max(output, 1)
            class_index = predicted.item()
            class_name = CLASS_NAMES[class_index]

            results.append((image_name, class_name))

    df = pd.DataFrame(results, columns=["image_filename", "predicted_label"])
    df.to_csv("LeModelSolutionB_TestResults.csv", index=False)
    print("✅ Submission CSV generated as 'LeModelSolutionB_TestResults.csv'.")

predict_and_generate_csv()


C:\Users\Gigabyte\AppData\Local\Temp\ipykernel_20704\3598730941.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location

✅ Submission CSV generated as 'LeModelSolutionB_TestResults.csv'.
